# Embedding Visualization — GNN Representations

This notebook visualizes the 256-dimensional embeddings produced by the trained
HGT (Heterogeneous Graph Transformer) model. We use UMAP to project embeddings
into 2D, then examine job, skill, and company clusters.

**Embeddings:** Produced by the HGT model trained for 100 epochs on MPS.
- Job embeddings: (481, 256)
- Skill embeddings: (427, 256)
- Company embeddings: (8, 256)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")

GRAPH_DIR = Path("..") / "data" / "graph"
DATA_DIR = Path("..") / "data" / "processed"
print("Graph directory:", GRAPH_DIR.resolve())

## 1. Load Embeddings

In [ ]:
# Load job embeddings (produced by model/embed.py)
job_embeddings = np.load(GRAPH_DIR / "job_embeddings.npy")
print(f"Job embeddings shape: {job_embeddings.shape}")

# Load mappings for labels
with open(GRAPH_DIR / "mappings.json") as f:
    mappings = json.load(f)

# Load the job DataFrame for role_family, company, seniority labels
df = pd.read_parquet(DATA_DIR / "jobs.parquet")
df = df.reset_index(drop=True)

print(f"Jobs in parquet: {len(df)}")
print(f"Job IDs in mappings: {len(mappings['job_ids'])}")
print(f"Skills in mappings: {len(mappings['skills'])}")
print(f"Companies in mappings: {len(mappings['companies'])}")
print(f"Role families: {sorted(df['role_family'].unique())}")
print(f"Seniority levels: {sorted(df['seniority'].unique())}")

In [ ]:
# Load skill and company embeddings from .npy files (learned representations from HGT)
skill_embeddings = np.load(GRAPH_DIR / "skill_embeddings.npy")
company_embeddings = np.load(GRAPH_DIR / "company_embeddings.npy")

print(f"Skill embeddings shape: {skill_embeddings.shape}")
print(f"Company embeddings shape: {company_embeddings.shape}")
print()
print(f"Job embedding norm (mean):     {np.linalg.norm(job_embeddings, axis=1).mean():.4f}")
print(f"Skill embedding norm (mean):   {np.linalg.norm(skill_embeddings, axis=1).mean():.4f}")
print(f"Company embedding norm (mean): {np.linalg.norm(company_embeddings, axis=1).mean():.4f}")

## 2. UMAP Projection — Jobs

We project the 256-dim job embeddings into 2D with UMAP, then color by three
different categorical attributes to see what structure the GNN learned.

In [ ]:
# Fit UMAP on job embeddings
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
job_2d = reducer.fit_transform(job_embeddings)

print(f"UMAP projection shape: {job_2d.shape}")

In [ ]:
# Color by role_family
role_families = df["role_family"].values[:len(job_2d)]
unique_roles = sorted(df["role_family"].dropna().unique())
role_palette = dict(zip(unique_roles, sns.color_palette("Set2", len(unique_roles))))

fig, ax = plt.subplots(figsize=(14, 10))
for role in unique_roles:
    mask = role_families == role
    ax.scatter(
        job_2d[mask, 0],
        job_2d[mask, 1],
        c=[role_palette[role]],
        label=role,
        alpha=0.6,
        s=20,
        edgecolors="none",
    )

ax.legend(title="Role Family", markerscale=3, fontsize=10)
ax.set_title("Job Embeddings — UMAP 2D Projection (colored by role_family)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

In [ ]:
# Color by company
companies = df["company"].values[:len(job_2d)]
unique_companies = sorted(df["company"].dropna().unique())
company_palette = dict(zip(unique_companies, sns.color_palette("tab10", len(unique_companies))))

fig, ax = plt.subplots(figsize=(14, 10))
for company in unique_companies:
    mask = companies == company
    ax.scatter(
        job_2d[mask, 0],
        job_2d[mask, 1],
        c=[company_palette[company]],
        label=company,
        alpha=0.6,
        s=20,
        edgecolors="none",
    )

ax.legend(title="Company", markerscale=3, fontsize=9, loc="best")
ax.set_title("Job Embeddings — UMAP 2D Projection (colored by company)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

In [ ]:
# Color by seniority
seniority_vals = df["seniority"].values[:len(job_2d)]
seniority_order = ["entry", "mid", "senior", "staff"]
seniority_palette = dict(zip(seniority_order, ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]))

fig, ax = plt.subplots(figsize=(14, 10))
for level in seniority_order:
    mask = seniority_vals == level
    if mask.sum() > 0:
        ax.scatter(
            job_2d[mask, 0],
            job_2d[mask, 1],
            c=[seniority_palette[level]],
            label=level,
            alpha=0.6,
            s=20,
            edgecolors="none",
        )

ax.legend(title="Seniority", markerscale=3, fontsize=10)
ax.set_title("Job Embeddings — UMAP 2D Projection (colored by seniority)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

## 3. Nearest Neighbors — Sample Jobs

For each of 5 sample jobs (one per role family), we find the 5 nearest neighbors
in embedding space using cosine distance. This demonstrates the quality of the
learned representations — similar jobs should cluster together.

In [ ]:
# Build a nearest-neighbor index on job embeddings
nn = NearestNeighbors(n_neighbors=6, metric="cosine")
nn.fit(job_embeddings)

# Pick 5 sample jobs spread across different role families, preferring jobs with titles
sample_indices = []
for role in unique_roles[:5]:
    role_mask = (df["role_family"] == role) & (df["title"].str.len() > 0)
    candidates = df[role_mask].index.tolist()
    if not candidates:
        # Fall back to any job in this role family
        candidates = df[df["role_family"] == role].index.tolist()
    if candidates:
        idx = candidates[0]
        if idx < len(job_embeddings):
            sample_indices.append(idx)

# If we have fewer than 5, fill with random jobs that have titles
rng = np.random.default_rng(42)
titled_indices = df[df["title"].str.len() > 0].index.tolist()
while len(sample_indices) < 5 and titled_indices:
    idx = rng.choice(titled_indices)
    if idx not in sample_indices and idx < len(job_embeddings):
        sample_indices.append(idx)
    titled_indices.remove(idx)

def job_label(idx):
    """Return a readable label for a job index."""
    if idx < len(df):
        title = df.iloc[idx]["title"]
        return title if title else f"[untitled] {df.iloc[idx]['role_family']}"
    return f"Job {idx}"

print("=" * 95)
for idx in sample_indices:
    distances, neighbors = nn.kneighbors(job_embeddings[idx:idx + 1])
    query_title = job_label(idx)
    query_company = df.iloc[idx]["company"] if idx < len(df) else "Unknown"
    query_role = df.iloc[idx]["role_family"] if idx < len(df) else "Unknown"

    print(f"\nQuery: {query_title} @ {query_company} [{query_role}]")
    print(f"{'Rank':<6} {'Cosine Dist':<14} {'Title':<40} {'Company':<20} {'Role'}")
    print("-" * 95)
    for rank, (dist, n_idx) in enumerate(zip(distances[0][1:], neighbors[0][1:]), 1):
        n_title = job_label(n_idx)
        n_company = df.iloc[n_idx]["company"] if n_idx < len(df) else "Unknown"
        n_role = df.iloc[n_idx]["role_family"] if n_idx < len(df) else "Unknown"
        print(f"  {rank:<4} {dist:<14.4f} {n_title:<40} {n_company:<20} {n_role}")
    print("=" * 95)

## 4. Skill Embedding Space

In [ ]:
# UMAP projection of skill embeddings
skill_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
skill_2d = skill_reducer.fit_transform(skill_embeddings)

skills_list = mappings["skills"]
skill_freqs = np.array(mappings.get("skill_freqs", [1] * len(skills_list)))

# Scale point size by frequency (log scale for readability)
sizes = np.log1p(skill_freqs) * 10

fig, ax = plt.subplots(figsize=(16, 12))
scatter = ax.scatter(
    skill_2d[:, 0],
    skill_2d[:, 1],
    s=sizes,
    c=np.log1p(skill_freqs),
    cmap="viridis",
    alpha=0.7,
    edgecolors="white",
    linewidths=0.3,
)
plt.colorbar(scatter, ax=ax, label="log(frequency + 1)")

# Label the top 30 most frequent skills
top_skill_indices = np.argsort(skill_freqs)[-30:]
for i in top_skill_indices:
    ax.annotate(
        skills_list[i],
        (skill_2d[i, 0], skill_2d[i, 1]),
        fontsize=8,
        fontweight="bold",
        alpha=0.9,
        ha="center",
        va="bottom",
    )

ax.set_title("Skill Embeddings — UMAP 2D Projection\n(size & color = frequency, top 30 labeled)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

## 5. Company Clusters

In [ ]:
# Company embeddings — with only 8 companies, we use a simple scatter + labels
# rather than UMAP (which needs n_neighbors < n_samples).
companies_list = mappings["companies"]
n_companies = len(companies_list)

# Count jobs per company from the dataframe
company_job_counts_dict = df["company"].value_counts().to_dict()
company_job_counts = np.array([company_job_counts_dict.get(c, 0) for c in companies_list])

if n_companies > 3:
    # With 8 companies, use UMAP with n_neighbors capped at n_companies - 1
    n_neighbors_company = min(5, n_companies - 1)
    company_reducer = umap.UMAP(
        n_components=2,
        random_state=42,
        n_neighbors=n_neighbors_company,
        min_dist=0.2,
    )
    company_2d = company_reducer.fit_transform(company_embeddings)
else:
    # Too few for UMAP — use first 2 PCA components
    from sklearn.decomposition import PCA
    company_2d = PCA(n_components=2).fit_transform(company_embeddings)

sizes = np.clip(company_job_counts * 15, 100, 800)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    company_2d[:, 0],
    company_2d[:, 1],
    s=sizes,
    c=company_job_counts,
    cmap="RdYlGn",
    alpha=0.8,
    edgecolors="grey",
    linewidths=1.0,
)
plt.colorbar(scatter, ax=ax, label="Number of Jobs")

# Label all companies
for i in range(n_companies):
    ax.annotate(
        f"{companies_list[i]} ({company_job_counts[i]})",
        (company_2d[i, 0], company_2d[i, 1]),
        fontsize=9,
        fontweight="bold",
        alpha=0.9,
        ha="center",
        va="bottom",
        xytext=(0, 8),
        textcoords="offset points",
    )

ax.set_title("Company Embeddings — UMAP 2D Projection\n(size & color = job count)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print("=== Embedding Summary ===")
print(f"Job embeddings:     {job_embeddings.shape[0]} vectors x {job_embeddings.shape[1]} dims")
print(f"Skill embeddings:   {skill_embeddings.shape[0]} vectors x {skill_embeddings.shape[1]} dims")
print(f"Company embeddings: {company_embeddings.shape[0]} vectors x {company_embeddings.shape[1]} dims")
print()
print(f"Job embedding norm (mean):     {np.linalg.norm(job_embeddings, axis=1).mean():.4f}")
print(f"Skill embedding norm (mean):   {np.linalg.norm(skill_embeddings, axis=1).mean():.4f}")
print(f"Company embedding norm (mean): {np.linalg.norm(company_embeddings, axis=1).mean():.4f}")